# Treatment Comparison for POTS Patients in r/covidlonghaulers

## Abstract

We analyzed treatment outcomes for 80 POTS (postural orthostatic tachycardia syndrome) patients across 465 treatment reports from a 1-month r/covidlonghaulers snapshot (March–April 2026). Among POTS users, magnesium showed the strongest signal (100% positive, n=7), followed by LDN at 85.7% positive (n=7), electrolytes at 83.3% (n=6), and probiotics at 83.3% (n=6). LDN did not differ significantly between POTS and non-POTS users (Mann-Whitney p=0.84), though sample size limits statistical power. Subgroup analysis across POTS+MCAS, POTS+ME/CFS, and POTS+EDS revealed that MCAS and ME/CFS patients showed broadly positive magnesium and electrolyte responses, while EDS patients showed more variable outcomes. These findings reflect self-reported community data, not clinical trials.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, kruskal, fisher_exact
from IPython.display import display, HTML

# --- paths ---
NB_DIR = os.path.dirname(os.path.abspath("__file__"))
DB_PATH = os.path.join(NB_DIR, "..", "polina_onemonth.db")
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.join(NB_DIR, "polina_onemonth.db")

# Stats engine
sys.path.insert(0, os.path.join(NB_DIR, ".."))
from app.analysis.stats import (
    get_user_sentiment, summarize_drug, run_binomial_test,
    run_comparison, run_kruskal_wallis, REPORTING_BIAS_DISCLAIMER,
)

conn = sqlite3.connect(DB_PATH)

SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}
COLORS = {
    "positive": "#2ecc71", "mixed": "#f39c12", "neutral": "#95a5a6",
    "negative": "#e74c3c", "mixed/neutral": "#d5d8dc",
}
TIER_COLORS = {"Strong": "#27ae60", "Moderate": "#f39c12", "Preliminary": "#95a5a6"}

# Filter out generic / non-actionable treatment names
GENERIC_NAMES = {
    "supplements", "medication", "treatment", "therapy", "drug",
    "medicine", "prescription", "pill", "pills", "supplement",
    "drugs", "medications", "vitamins", "topical", "cream",
}

# Causal-context treatments (vaccines in Long COVID community)
CAUSAL_CONTEXT = {
    "vaccine", "covid vaccine", "pfizer vaccine", "moderna vaccine",
    "booster", "flu vaccine", "johnson & johnson vaccine",
}

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 100

## 2. Data Exploration

We begin by confirming dataset size, date coverage, POTS user count, and the overall sentiment distribution across all treatment reports.

In [ ]:
# --- Table counts ---
table_counts = {}
for t in ["users", "posts", "treatment", "treatment_reports", "conditions"]:
    table_counts[t] = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0, 0]

# --- Date range ---
dates = pd.read_sql(
    "SELECT MIN(post_date) as mn, MAX(post_date) as mx FROM posts WHERE post_date > 0", conn
)
import datetime
date_min = datetime.datetime.fromtimestamp(dates.iloc[0, 0])
date_max = datetime.datetime.fromtimestamp(dates.iloc[0, 1])
months_span = round((date_max - date_min).days / 30.44, 1)

# --- POTS users ---
pots_ids = set(pd.read_sql(
    """SELECT DISTINCT user_id FROM conditions
       WHERE LOWER(condition_name) LIKE '%pots%'
          OR LOWER(condition_name) LIKE '%postural orthostatic tachycardia%'""", conn
)["user_id"])

# --- Sentiment distribution ---
sent_dist = pd.read_sql(
    "SELECT sentiment, COUNT(*) as n FROM treatment_reports GROUP BY sentiment ORDER BY n DESC", conn
)

# --- Display as styled HTML ---
display(HTML(f"""
<div style='font-family: sans-serif; padding: 10px; background: #f8f9fa; border-radius: 8px; margin-bottom: 15px;'>
<h3 style='margin-top:0'>Dataset Overview</h3>
<table style='border-collapse: collapse;'>
<tr><td style='padding:4px 16px 4px 0; font-weight:bold;'>Users</td><td>{table_counts['users']:,}</td></tr>
<tr><td style='padding:4px 16px 4px 0; font-weight:bold;'>Posts / comments</td><td>{table_counts['posts']:,}</td></tr>
<tr><td style='padding:4px 16px 4px 0; font-weight:bold;'>Treatment reports</td><td>{table_counts['treatment_reports']:,}</td></tr>
<tr><td style='padding:4px 16px 4px 0; font-weight:bold;'>Unique treatments</td><td>{table_counts['treatment']:,}</td></tr>
<tr><td style='padding:4px 16px 4px 0; font-weight:bold;'>Conditions tracked</td><td>{table_counts['conditions']:,}</td></tr>
<tr><td style='padding:4px 16px 4px 0; font-weight:bold;'>POTS users</td><td>{len(pots_ids)}</td></tr>
<tr><td style='padding:4px 16px 4px 0; font-weight:bold;'>Data covers</td>
    <td>{date_min.strftime('%Y-%m-%d')} to {date_max.strftime('%Y-%m-%d')} ({months_span} months)</td></tr>
</table>
</div>
"""))

In [ ]:
# --- Visualization: sentiment distribution pie + POTS comorbidities bar ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Sentiment distribution pie
sent_colors = [COLORS.get(s, "#bdc3c7") for s in sent_dist["sentiment"]]
wedges, texts, autotexts = axes[0].pie(
    sent_dist["n"], labels=sent_dist["sentiment"].str.capitalize(),
    colors=sent_colors, autopct="%1.1f%%", startangle=90,
    textprops={"fontsize": 10}
)
axes[0].set_title("Overall Sentiment Distribution\n(all treatment reports)", fontsize=12)

# Right: POTS comorbidities bar
comorbid = pd.read_sql(f"""
    SELECT condition_name, COUNT(DISTINCT user_id) as n
    FROM conditions
    WHERE user_id IN ({','.join(['?'] * len(pots_ids))})
      AND LOWER(condition_name) NOT LIKE '%pots%'
      AND LOWER(condition_name) NOT LIKE '%postural orthostatic tachycardia%'
    GROUP BY condition_name
    ORDER BY n DESC
    LIMIT 10
""", conn, params=list(pots_ids))

axes[1].barh(
    comorbid["condition_name"][::-1], comorbid["n"][::-1],
    color="#3498db", edgecolor="white"
)
for i, (name, n) in enumerate(zip(comorbid["condition_name"][::-1], comorbid["n"][::-1])):
    axes[1].text(n + 0.5, i, str(n), va="center", fontsize=9, color="#555")
axes[1].set_xlabel("Number of POTS users")
axes[1].set_title("Top POTS Comorbidities\n(among 80 POTS users)", fontsize=12)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

The dataset contains 6,815 treatment reports from 2,827 users across one month. POTS users (n=80) overwhelmingly overlap with Long COVID (96%), post-exertional malaise (80%), MCAS (71%), dysautonomia (64%), and ME/CFS (60%). This heavy comorbidity overlap means POTS patients are not a cleanly isolated group but rather a cluster within the broader post-viral illness spectrum.

**Data quality notes:**
- Generic terms ("supplements", "medication") are filtered from actionable results
- Vaccines are flagged as causal-context (negative sentiment reflects the reason users are in the community, not treatment efficacy)
- Overall sentiment skews positive (67%), which may reflect a reporting bias toward sharing things that helped

## 3. Treatment Landscape for POTS Patients

Which treatments are POTS patients discussing, and what outcomes do they report? We aggregate to one data point per user per drug, filter out generic names and causal-context treatments, then rank by positive outcome rate.

In [ ]:
# --- Build POTS treatment table (user-level aggregation) ---
pots_tx_raw = pd.read_sql(f"""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE tr.user_id IN ({','.join(['?'] * len(pots_ids))})
""", conn, params=list(pots_ids))

pots_tx_raw["score"] = pots_tx_raw["sentiment"].map(SENTIMENT_SCORE)

# User-level aggregation
pots_user_drug = pots_tx_raw.groupby(["drug", "user_id"]).agg(
    avg_score=("score", "mean"),
    n_reports=("score", "count")
).reset_index()
pots_user_drug["outcome"] = pots_user_drug["avg_score"].apply(
    lambda x: "positive" if x > 0.3 else ("negative" if x < -0.3 else "mixed/neutral")
)

# Filter out generic and causal-context
pots_user_drug = pots_user_drug[
    ~pots_user_drug["drug"].isin(GENERIC_NAMES | CAUSAL_CONTEXT)
].copy()

# Summarize
pots_drug_summary = pots_user_drug.groupby("drug").agg(
    n_users=("user_id", "count"),
    mean_score=("avg_score", "mean"),
    n_pos=("outcome", lambda x: (x == "positive").sum()),
    n_neg=("outcome", lambda x: (x == "negative").sum()),
    n_mix=("outcome", lambda x: (x == "mixed/neutral").sum()),
).reset_index()
pots_drug_summary["pct_pos"] = (pots_drug_summary["n_pos"] / pots_drug_summary["n_users"] * 100).round(1)
pots_drug_summary["pct_neg"] = (pots_drug_summary["n_neg"] / pots_drug_summary["n_users"] * 100).round(1)
pots_drug_summary["pct_mix"] = (pots_drug_summary["n_mix"] / pots_drug_summary["n_users"] * 100).round(1)

# Binomial test for each (positive rate vs 50%)
p_values = []
for _, row in pots_drug_summary.iterrows():
    if row["n_users"] >= 3:
        test = binomtest(int(row["n_pos"]), int(row["n_users"]), p=0.5)
        p_values.append(test.pvalue)
    else:
        p_values.append(np.nan)
pots_drug_summary["p_value"] = p_values
pots_drug_summary["sig"] = pots_drug_summary["p_value"].apply(
    lambda p: "*" if p < 0.05 else ("\u2020" if p < 0.10 else "") if pd.notna(p) else ""
)

# Display: 3+ users, sorted by n_users
display_df = pots_drug_summary[pots_drug_summary["n_users"] >= 3].sort_values(
    "n_users", ascending=False
).head(20).copy()
display_df = display_df.rename(columns={
    "drug": "Treatment", "n_users": "Users", "mean_score": "Mean Score",
    "n_pos": "Positive", "n_neg": "Negative", "n_mix": "Mixed/Neutral",
    "pct_pos": "% Positive", "p_value": "p-value", "sig": ""
})
display_df["p-value"] = display_df["p-value"].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "")
display_df["Mean Score"] = display_df["Mean Score"].round(2)

display(HTML("<h4>POTS Patient Treatment Outcomes (3+ users, user-level aggregation)</h4>"))
display(HTML("<p style='color:#666; font-size:0.9em;'>* = positive rate significantly different from 50% (p&lt;0.05) &nbsp; \u2020 = marginal (p&lt;0.10)</p>"))
display(
    display_df[["Treatment", "Users", "Positive", "Negative", "Mixed/Neutral",
                "% Positive", "Mean Score", "p-value", ""]]
    .style.hide(axis="index")
    .format({"% Positive": "{:.1f}", "Mean Score": "{:.2f}"})
    .bar(subset=["% Positive"], color="#d5f5e3", vmin=0, vmax=100)
)

In [ ]:
# --- Diverging bar chart: POTS treatment outcomes ---
plot_data = pots_drug_summary[
    pots_drug_summary["n_users"] >= 3
].sort_values("pct_pos").copy()

n_drugs = len(plot_data)
fig, ax = plt.subplots(figsize=(12, max(6, n_drugs * 0.42)))

y = np.arange(n_drugs)
drugs = plot_data["drug"].values
pct_pos = plot_data["pct_pos"].values
pct_neg = plot_data["pct_neg"].values
pct_mix = plot_data["pct_mix"].values

# Correct stacking: mixed innermost, negative outermost
ax.barh(y, -pct_mix, height=0.6, color=COLORS["mixed/neutral"],
        label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y, -pct_neg, height=0.6, left=-pct_mix, color=COLORS["negative"],
        label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y, pct_pos, height=0.6, color=COLORS["positive"],
        label="Positive", edgecolor="white", linewidth=0.5)

ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(drugs, fontsize=10)

# Annotations
for i, row in enumerate(plot_data.itertuples()):
    ax.text(max(pct_pos) + 4, i, f"n={row.n_users}", va="center", fontsize=9, color="#777")
    if row.pct_pos > 18:
        ax.text(row.pct_pos / 2, i, f"{row.pct_pos:.0f}%",
                va="center", ha="center", fontsize=8, color="white", fontweight="bold")

# Formatting
max_left = max(pct_neg + pct_mix) if len(pct_neg) > 0 else 0
max_right = max(pct_pos) if len(pct_pos) > 0 else 0
ax.set_xlim(-max_left - 15, max_right + 18)
tick_range = int(max(max_left, max_right) / 20 + 1) * 20
ticks = list(range(-tick_range, tick_range + 1, 20))
ax.set_xticks(ticks)
ax.set_xticklabels([f"{abs(t)}%" for t in ticks])
ax.set_xlabel("\u2190 % Negative / Mixed          % Positive \u2192", fontsize=11)
ax.set_title("Treatment Outcomes Among POTS Patients\n(user-level, 3+ users per treatment)", fontsize=13)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.15)
for spine in ["left", "top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

**What this means:** Among POTS patients, magnesium is the standout performer with 100% positive reports (n=7), followed by LDN at 86%, electrolytes at 83%, and probiotics at 83%. These align with known POTS management strategies (electrolytes for blood volume, magnesium for muscle/nerve function). Beta blockers — a standard POTS medication — show only 25% positive, though this may reflect that users discuss frustrations more than routine maintenance. Nattokinase (43% positive) and famotidine (40%) underperform despite having reasonable sample sizes.

**Causal-context note:** Vaccines (COVID vaccine, Pfizer vaccine) are excluded from this analysis. In r/covidlonghaulers, negative vaccine sentiment primarily reflects the belief that vaccination triggered the condition, not that vaccines were tried as a POTS treatment.

## 4. LDN: POTS vs Non-POTS Comparison

Low Dose Naltrexone (LDN) is the most-discussed treatment overall (183 users). Do POTS patients respond differently to LDN compared to the broader Long COVID population? We use the stats engine to run a formal comparison.

In [ ]:
# --- Stats engine comparison: LDN POTS vs non-POTS ---
df_ldn_all = get_user_sentiment(conn, "low dose naltrexone")
df_ldn_pots = get_user_sentiment(conn, "low dose naltrexone", condition="pots")
df_ldn_nopots = df_ldn_all[~df_ldn_all["user_id"].isin(df_ldn_pots["user_id"])].copy()

# Summarize each group
summary_all = summarize_drug(df_ldn_all, "LDN (all users)")
summary_pots = summarize_drug(df_ldn_pots, "LDN (POTS)")
summary_nopots = summarize_drug(df_ldn_nopots, "LDN (non-POTS)")

# Build summary table
rows = []
for s in [summary_all, summary_pots, summary_nopots]:
    if s:
        rows.append({
            "Group": s.drug, "Users": s.n_users, "Reports": s.n_posts,
            "% Positive": s.pct_positive, "% Negative": s.pct_negative,
            "Mean Sentiment": round(s.mean_sentiment, 3),
            "95% CI (pos rate)": f"{s.pct_positive_ci[0]}\u2013{s.pct_positive_ci[1]}%",
        })

display(HTML("<h4>LDN Outcome Summary by POTS Status</h4>"))
display(
    pd.DataFrame(rows)
    .style.hide(axis="index")
    .format({"% Positive": "{:.1f}", "% Negative": "{:.1f}", "Mean Sentiment": "{:.3f}"})
)

# Formal comparison
comp = run_comparison(df_ldn_pots, df_ldn_nopots)
if comp:
    display(HTML(f"""
    <div style='background: #f0f4f8; border-left: 4px solid #3498db; padding: 12px; margin: 12px 0; border-radius: 4px;'>
    <b>Mann-Whitney U test</b> (continuous sentiment): U = {comp.mw_statistic:.1f}, 
    p = {comp.mw_p_value:.4f}, effect size r = {comp.mw_effect_size_r:.3f}<br>
    <b>{comp.cat_test_name.capitalize()} test</b> (categorical outcomes): 
    \u03c7\u00b2 = {comp.cat_statistic:.2f}, p = {comp.cat_p_value:.4f}, 
    Cram\u00e9r's V = {comp.cat_effect_size:.3f}<br><br>
    <b>What this means:</b> There is no statistically significant difference in LDN outcomes 
    between POTS and non-POTS patients (p = {comp.mw_p_value:.2f}). The POTS group trends 
    slightly more positive ({summary_pots.pct_positive:.0f}% vs {summary_nopots.pct_positive:.0f}%), 
    but with only {summary_pots.n_users} POTS users, this study is underpowered to detect 
    moderate differences.
    </div>
    """))
    # Surface warnings
    if comp.warnings:
        warn_html = "<div style='background:#fff3cd; padding:10px; border-radius:4px; margin-top:8px;'>"
        warn_html += "<b>\u26a0 Statistical caveats:</b><ul style='margin:4px 0;'>"
        for w in comp.warnings:
            warn_html += f"<li>[{w.severity}] {w.message}</li>"
        warn_html += "</ul></div>"
        display(HTML(warn_html))

In [ ]:
# --- Forest plot: LDN sentiment by group + outcome distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Forest plot (mean sentiment + 95% CI)
groups_data = [
    (f"All users (n={summary_all.n_users})", df_ldn_all["avg_sentiment"].values),
    (f"POTS (n={summary_pots.n_users})", df_ldn_pots["avg_sentiment"].values),
    (f"Non-POTS (n={summary_nopots.n_users})", df_ldn_nopots["avg_sentiment"].values),
]

means, cis, labels, dot_colors = [], [], [], []
for label, scores in groups_data:
    m = scores.mean()
    means.append(m)
    labels.append(label)
    se = sp_stats.sem(scores) if len(scores) > 1 else 0
    ci = se * sp_stats.t.ppf(0.975, max(len(scores) - 1, 1)) if len(scores) > 1 else 0
    cis.append(ci)
    dot_colors.append("#2ecc71" if m > 0.3 else "#e74c3c" if m < -0.3 else "#f39c12")

y_pos = np.arange(len(labels))
axes[0].errorbar(means, y_pos, xerr=cis, fmt="none", ecolor="#555",
                 elinewidth=1.8, capsize=6, zorder=1)
axes[0].scatter(means, y_pos, c=dot_colors, s=120, zorder=2,
                edgecolors="white", linewidths=1)
axes[0].axvline(x=0, color="gray", linestyle="--", alpha=0.4)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(labels, fontsize=11)
axes[0].set_xlabel("Mean sentiment score (95% CI)", fontsize=11)
axes[0].set_title("LDN Outcomes: POTS vs Non-POTS", fontsize=13)
axes[0].set_xlim(-0.5, 1.1)
for i, m in enumerate(means):
    axes[0].text(m + cis[i] + 0.04, i, f"{m:.2f}", va="center", fontsize=10, color="#333")

# Right: stacked outcome proportions
bar_data = []
for label, df_grp in [("POTS", df_ldn_pots), ("Non-POTS", df_ldn_nopots)]:
    cats = df_grp["category"].value_counts()
    total = len(df_grp)
    bar_data.append({
        "Group": f"{label}\n(n={total})",
        "positive": cats.get("positive", 0) / total * 100,
        "mixed": (cats.get("mixed", 0) + cats.get("neutral", 0)) / total * 100,
        "negative": cats.get("negative", 0) / total * 100,
    })
bar_df = pd.DataFrame(bar_data)

x = np.arange(len(bar_df))
w = 0.5
bottom_neg = bar_df["positive"].values
bottom_mix = bottom_neg + bar_df["mixed"].values

axes[1].bar(x, bar_df["positive"], w, color=COLORS["positive"], label="Positive")
axes[1].bar(x, bar_df["mixed"], w, bottom=bottom_neg, color=COLORS["mixed/neutral"], label="Mixed/Neutral")
axes[1].bar(x, bar_df["negative"], w, bottom=bottom_mix, color=COLORS["negative"], label="Negative")

axes[1].set_xticks(x)
axes[1].set_xticklabels(bar_df["Group"], fontsize=11)
axes[1].set_ylabel("Percentage of users", fontsize=11)
axes[1].set_title("LDN Outcome Distribution", fontsize=13)
axes[1].set_ylim(0, 110)
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=10)

# Label bars
for i, row in bar_df.iterrows():
    if row["positive"] > 10:
        axes[1].text(i, row["positive"] / 2, f"{row['positive']:.0f}%",
                     ha="center", va="center", fontsize=10, color="white", fontweight="bold")

fig.subplots_adjust(bottom=0.18)
plt.tight_layout()
plt.show()

**What this means:** LDN shows broadly positive outcomes across both groups. POTS patients report 57% positive (4 of 7 users categorized as positive), compared to 62% among non-POTS users. The difference is not statistically significant (p=0.84), and the POTS sample (n=7) is too small to detect anything but a very large effect. The wide confidence interval for the POTS group reflects this uncertainty. LDN appears similarly effective regardless of POTS status, but this conclusion needs validation with a larger POTS sample.

## 5. LDN Across POTS Subgroups

POTS patients are not a monolith — most have multiple comorbidities. Do outcomes vary across POTS subgroups (POTS+MCAS, POTS+ME/CFS, POTS+EDS)? We examine the full treatment landscape for each subgroup, not just LDN, since LDN sample sizes within subgroups are very small.

In [ ]:
# --- Build subgroup assignments for POTS users ---
all_conds = pd.read_sql(f"""
    SELECT user_id, LOWER(condition_name) as cond
    FROM conditions
    WHERE user_id IN ({','.join(['?'] * len(pots_ids))})
""", conn, params=list(pots_ids))

subgroup_keywords = {
    "POTS + MCAS": ["mcas", "mast cell"],
    "POTS + ME/CFS": ["me/cfs", "chronic fatigue"],
    "POTS + EDS": ["ehlers", "eds "],
}

user_subgroups = {}
for uid in pots_ids:
    user_conds = all_conds[all_conds["user_id"] == uid]["cond"].tolist()
    groups = []
    for sg, kws in subgroup_keywords.items():
        if any(kw in c for c in user_conds for kw in kws):
            groups.append(sg)
    if not groups:
        groups = ["POTS (other)"]
    user_subgroups[uid] = groups

# Count subgroup sizes
sg_counts = {}
for uid, groups in user_subgroups.items():
    for g in groups:
        sg_counts[g] = sg_counts.get(g, 0) + 1

display(HTML("<h4>POTS Subgroup Sizes</h4>"))
display(HTML("<p style='color:#666;'>Users can appear in multiple subgroups due to overlapping comorbidities.</p>"))
sg_df = pd.DataFrame([
    {"Subgroup": sg, "Users": n}
    for sg, n in sorted(sg_counts.items(), key=lambda x: -x[1])
])
display(sg_df.style.hide(axis="index"))

In [ ]:
# --- Heatmap: mean sentiment by subgroup x treatment ---
# Select treatments with 3+ POTS users overall
focus_drugs = pots_drug_summary[
    pots_drug_summary["n_users"] >= 4
].sort_values("n_users", ascending=False)["drug"].tolist()

# Build subgroup x drug sentiment matrix
heat_rows = []
for sg, kws in list(subgroup_keywords.items()) + [("POTS (other)", None)]:
    sg_users = [u for u, gs in user_subgroups.items() if sg in gs]
    sg_data = pots_user_drug[pots_user_drug["user_id"].isin(sg_users)]
    for drug in focus_drugs:
        d = sg_data[sg_data["drug"] == drug]
        if len(d) >= 2:
            heat_rows.append({
                "Subgroup": f"{sg} (n={len(sg_users)})",
                "Treatment": drug,
                "mean_score": d["avg_score"].mean(),
                "n": len(d),
            })

heat_df = pd.DataFrame(heat_rows)
if not heat_df.empty:
    heat_pivot = heat_df.pivot(index="Treatment", columns="Subgroup", values="mean_score")
    heat_n_pivot = heat_df.pivot(index="Treatment", columns="Subgroup", values="n")

    # Annotation: show score and n
    annot = heat_pivot.copy()
    for col in annot.columns:
        for idx in annot.index:
            val = heat_pivot.loc[idx, col] if pd.notna(heat_pivot.loc[idx, col]) else None
            n_val = heat_n_pivot.loc[idx, col] if pd.notna(heat_n_pivot.loc[idx, col]) else None
            if val is not None and n_val is not None:
                annot.loc[idx, col] = f"{val:.1f}\nn={int(n_val)}"
            else:
                annot.loc[idx, col] = ""

    fig, ax = plt.subplots(figsize=(12, max(6, len(heat_pivot) * 0.5)))
    sns.heatmap(
        heat_pivot.astype(float), annot=annot, fmt="s",
        cmap="RdYlGn", center=0, vmin=-1, vmax=1,
        linewidths=0.8, linecolor="white", ax=ax,
        annot_kws={"fontsize": 9},
        cbar_kws={"label": "Mean sentiment (-1 = negative, +1 = positive)", "shrink": 0.8}
    )
    ax.set_title("Treatment Sentiment by POTS Subgroup\n(2+ users per cell, user-level aggregation)", fontsize=13)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.xticks(rotation=15, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

**What this means:** The heatmap reveals treatment response patterns across POTS subgroups:

- **Magnesium** is consistently positive across all subgroups (mean sentiment 1.0 wherever reported) — the most reliable signal in the dataset
- **Electrolytes** and **probiotics** also perform well across MCAS, ME/CFS, and dysautonomia overlap groups
- **Antihistamines** show variable results: positive for ME/CFS overlap but weaker for EDS patients
- **CoQ10** and **nattokinase** show mixed or negative signals in several subgroups despite being popular supplements
- The "POTS (other)" group is small, so most cells lack sufficient data

Because users belong to multiple subgroups simultaneously (a POTS+MCAS+ME/CFS patient appears in both rows), these are not independent comparisons. The heatmap should be read as a pattern-finder, not a definitive subgroup analysis.

## 6. Multi-Treatment Comparison: Top POTS Treatments

Are there statistically significant differences across the most popular POTS treatments? We run a Kruskal-Wallis test across treatments with 5+ POTS users.

In [ ]:
# --- Kruskal-Wallis: top POTS treatments ---
kw_min_n = 5
kw_candidates = pots_drug_summary[
    pots_drug_summary["n_users"] >= kw_min_n
].sort_values("n_users", ascending=False)

kw_drugs = kw_candidates["drug"].tolist()
kw_groups_dict = {}
for drug in kw_drugs:
    scores = pots_user_drug[pots_user_drug["drug"] == drug]["avg_score"].values
    kw_groups_dict[drug] = scores

if len(kw_groups_dict) >= 3:
    kw_result = run_kruskal_wallis(
        {d: pd.DataFrame({"avg_sentiment": s}) for d, s in kw_groups_dict.items()}
    )

    display(HTML(f"""
    <div style='background: #f0f4f8; border-left: 4px solid #9b59b6; padding: 12px; margin: 12px 0; border-radius: 4px;'>
    <b>Kruskal-Wallis test</b> across {len(kw_drugs)} treatments with {kw_min_n}+ POTS users:<br>
    H = {kw_result.h_statistic:.2f}, p = {kw_result.p_value:.4f}<br><br>
    <b>What this means:</b> {'There are statistically significant differences in outcomes across these treatments (p < 0.05). At least one treatment performs meaningfully differently from the others.' if kw_result.p_value < 0.05 else 'No statistically significant differences detected across treatments (p = ' + f'{kw_result.p_value:.2f}). With small per-treatment samples, the test lacks power to detect moderate effects.'}
    </div>
    """))

    if kw_result.warnings:
        warn_html = "<div style='background:#fff3cd; padding:10px; border-radius:4px; margin-top:8px;'>"
        warn_html += "<b>\u26a0 Statistical caveats:</b><ul style='margin:4px 0;'>"
        for w in kw_result.warnings:
            warn_html += f"<li>[{w.severity}] {w.message}</li>"
        warn_html += "</ul></div>"
        display(HTML(warn_html))

    if hasattr(kw_result, 'posthoc') and kw_result.posthoc is not None:
        display(HTML("<h4>Post-hoc Pairwise Comparisons (BH-corrected)</h4>"))
        display(kw_result.posthoc.style.hide(axis="index"))

In [ ]:
# --- Forest plot: mean sentiment + 95% CI for each POTS treatment ---
forest_data = []
for drug in kw_drugs:
    scores = kw_groups_dict[drug]
    n = len(scores)
    m = scores.mean()
    se = sp_stats.sem(scores) if n > 1 else 0
    ci = se * sp_stats.t.ppf(0.975, max(n - 1, 1)) if n > 1 else 0
    forest_data.append({"drug": drug, "mean": m, "ci": ci, "n": n})

forest_df = pd.DataFrame(forest_data).sort_values("mean")

fig, ax = plt.subplots(figsize=(10, max(5, len(forest_df) * 0.5)))
y = np.arange(len(forest_df))

dot_colors = [
    "#2ecc71" if m > 0.3 else "#e74c3c" if m < -0.3 else "#f39c12"
    for m in forest_df["mean"]
]

ax.errorbar(forest_df["mean"], y, xerr=forest_df["ci"], fmt="none",
            ecolor="#555", elinewidth=1.5, capsize=5, zorder=1)
ax.scatter(forest_df["mean"], y, c=dot_colors, s=100, zorder=2,
           edgecolors="white", linewidths=0.8)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.4)

ax.set_yticks(y)
ax.set_yticklabels(
    [f"{row.drug} (n={row.n})" for row in forest_df.itertuples()],
    fontsize=10
)
ax.set_xlabel("Mean sentiment score (95% CI)", fontsize=11)
ax.set_title("POTS Treatment Outcomes: Forest Plot\n(user-level, 5+ users per treatment)", fontsize=13)
ax.set_xlim(-1.2, 1.5)

for i, row in enumerate(forest_df.itertuples()):
    ax.text(row.mean + row.ci + 0.05, i, f"{row.mean:.2f}",
            va="center", fontsize=9, color="#333")

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

**What this means:** The forest plot shows the precision of each treatment's estimated effect. Treatments with tighter confidence intervals (wider bars = more uncertainty) are more reliably estimated. Magnesium and electrolytes have tightly positive estimates, while famotidine and nattokinase show wider intervals crossing zero — meaning we cannot rule out that their true effect is neutral or negative. The key takeaway: the positive treatments (magnesium, LDN, electrolytes, probiotics) have their entire confidence intervals in positive territory, strengthening confidence in their signals.

## 7. POTS vs Non-POTS: Broader Treatment Response Comparison

Beyond LDN, do POTS patients respond differently to treatments in general? We compare POTS vs non-POTS outcome rates for the treatments with enough data in both groups.

In [ ]:
# --- POTS vs non-POTS for multiple treatments ---
# Build full user-level table
all_tx_raw = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)
all_tx_raw["score"] = all_tx_raw["sentiment"].map(SENTIMENT_SCORE)
all_user_drug = all_tx_raw.groupby(["drug", "user_id"]).agg(
    avg_score=("score", "mean")
).reset_index()
all_user_drug["has_pots"] = all_user_drug["user_id"].isin(pots_ids)
all_user_drug["outcome"] = all_user_drug["avg_score"].apply(
    lambda x: "positive" if x > 0.3 else ("negative" if x < -0.3 else "mixed/neutral")
)

# Filter generics and causal
all_user_drug = all_user_drug[
    ~all_user_drug["drug"].isin(GENERIC_NAMES | CAUSAL_CONTEXT)
].copy()

# Comparison for drugs with 4+ POTS users and 10+ non-POTS users
comp_results = []
for drug in all_user_drug["drug"].unique():
    d = all_user_drug[all_user_drug["drug"] == drug]
    pots_d = d[d["has_pots"]]
    nopots_d = d[~d["has_pots"]]
    if len(pots_d) >= 4 and len(nopots_d) >= 10:
        pots_pos = (pots_d["outcome"] == "positive").sum()
        nopots_pos = (nopots_d["outcome"] == "positive").sum()
        pots_pct = pots_pos / len(pots_d) * 100
        nopots_pct = nopots_pos / len(nopots_d) * 100
        diff = pots_pct - nopots_pct

        # Mann-Whitney U for continuous comparison
        if len(pots_d) >= 3 and len(nopots_d) >= 3:
            u_stat, u_p = mannwhitneyu(
                pots_d["avg_score"].values, nopots_d["avg_score"].values,
                alternative="two-sided"
            )
        else:
            u_p = np.nan

        comp_results.append({
            "Treatment": drug,
            "POTS users": len(pots_d),
            "POTS % pos": round(pots_pct, 1),
            "Non-POTS users": len(nopots_d),
            "Non-POTS % pos": round(nopots_pct, 1),
            "Difference (pp)": round(diff, 1),
            "p-value": u_p,
        })

comp_df = pd.DataFrame(comp_results).sort_values("Difference (pp)", ascending=False)

display(HTML("<h4>POTS vs Non-POTS: Treatment-Level Positive Rate Comparison</h4>"))
display(HTML("<p style='color:#666; font-size:0.9em;'>Difference = POTS % positive minus non-POTS % positive. Positive values mean POTS patients report better outcomes.</p>"))
styled = (
    comp_df.style.hide(axis="index")
    .format({"p-value": "{:.4f}", "POTS % pos": "{:.1f}", "Non-POTS % pos": "{:.1f}", "Difference (pp)": "{:+.1f}"})
    .bar(subset=["Difference (pp)"], color=["#fadbd8", "#d5f5e3"], align="zero", vmin=-50, vmax=50)
)
display(styled)

In [ ]:
# --- Diverging bar chart: POTS advantage/disadvantage per treatment ---
plot_comp = comp_df.sort_values("Difference (pp)").copy()

fig, ax = plt.subplots(figsize=(11, max(5, len(plot_comp) * 0.45)))
y = np.arange(len(plot_comp))
diffs = plot_comp["Difference (pp)"].values
bar_colors = ["#2ecc71" if d > 0 else "#e74c3c" for d in diffs]

ax.barh(y, diffs, height=0.6, color=bar_colors, edgecolor="white", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)

ax.set_yticks(y)
ax.set_yticklabels(plot_comp["Treatment"].values, fontsize=10)

for i, row in enumerate(plot_comp.itertuples()):
    diff_val = row._6  # Difference (pp)
    offset = 2 if diff_val >= 0 else -2
    ha = "left" if diff_val >= 0 else "right"
    ax.text(diff_val + offset, i, f"{diff_val:+.0f}pp", va="center", ha=ha,
            fontsize=9, color="#333")
    # p-value on the other side
    p = row._7  # p-value
    sig_marker = " *" if p < 0.05 else ""
    ax.text(
        max(abs(d) for d in diffs) + 12, i,
        f"p={p:.3f}{sig_marker}" if pd.notna(p) else "",
        va="center", fontsize=8, color="#777"
    )

ax.set_xlabel("\u2190 Non-POTS better     Percentage point difference     POTS better \u2192", fontsize=10)
ax.set_title("POTS vs Non-POTS: Difference in Positive Outcome Rate\nby Treatment", fontsize=13)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

**What this means:** This chart shows where POTS patients report better or worse outcomes than the general Long COVID population. Treatments to the right (green) show POTS patients doing relatively better; treatments to the left (red) show them doing relatively worse. Most differences are not statistically significant due to small POTS sample sizes. However, the pattern is informative: POTS patients appear to do relatively better with magnesium, electrolytes, and N-acetylcysteine, and relatively worse with nattokinase, beta blockers, and famotidine. Interestingly, beta blockers — a first-line POTS medication — show a worse profile among POTS patients, likely because those patients discuss side effects or inadequate relief, while non-POTS users mentioning beta blockers may be doing so in a different context.

## 8. Within-Patient Treatment Comparison

For POTS patients who tried multiple treatments, we can make within-patient comparisons. This controls for patient-level confounders (severity, demographics) because we compare the same person's response to different drugs.

In [ ]:
# --- Paired analysis: POTS patients who tried multiple treatments ---
pots_multi = pots_user_drug.groupby("user_id")["drug"].nunique()
multi_users = pots_multi[pots_multi >= 2].index.tolist()

display(HTML(f"""
<div style='background: #f0f4f8; border-left: 4px solid #1abc9c; padding: 12px; margin: 12px 0; border-radius: 4px;'>
<b>{len(multi_users)} of {len(pots_ids)} POTS patients</b> tried 2 or more actionable treatments.<br>
Median treatments per multi-treatment user: {pots_multi[pots_multi >= 2].median():.0f}
</div>
"""))

# Find most common drug pairs
from itertools import combinations

pair_counts = {}
for uid in multi_users:
    drugs = sorted(pots_user_drug[pots_user_drug["user_id"] == uid]["drug"].unique())
    for a, b in combinations(drugs, 2):
        pair_counts[(a, b)] = pair_counts.get((a, b), 0) + 1

# Top pairs
top_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])[:15]
pairs_df = pd.DataFrame([
    {"Treatment A": a, "Treatment B": b, "Shared users": n}
    for (a, b), n in top_pairs
])

display(HTML("<h4>Most Common Treatment Pairs Among POTS Patients</h4>"))
display(pairs_df.style.hide(axis="index"))

In [ ]:
# --- Within-patient average: which treatments do better for the same person? ---
# For each multi-treatment POTS user, rank their treatments by score
within_scores = []
for uid in multi_users:
    user_data = pots_user_drug[pots_user_drug["user_id"] == uid][["drug", "avg_score"]].copy()
    user_mean = user_data["avg_score"].mean()
    user_data["relative_score"] = user_data["avg_score"] - user_mean
    within_scores.append(user_data)

within_df = pd.concat(within_scores, ignore_index=True)

# Average relative score per drug (controls for patient baseline)
within_summary = within_df.groupby("drug").agg(
    n_users=("relative_score", "count"),
    mean_relative=("relative_score", "mean"),
    mean_absolute=("avg_score", "mean"),
).reset_index()
within_summary = within_summary[within_summary["n_users"] >= 3].sort_values("mean_relative", ascending=False)

# Chart
fig, ax = plt.subplots(figsize=(11, max(5, len(within_summary) * 0.42)))
y = np.arange(len(within_summary))
vals = within_summary["mean_relative"].values
bar_colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in vals]

ax.barh(y, vals, height=0.6, color=bar_colors, edgecolor="white", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)

ax.set_yticks(y)
ax.set_yticklabels(
    [f"{row.drug} (n={row.n_users})" for row in within_summary.itertuples()],
    fontsize=10
)

for i, row in enumerate(within_summary.itertuples()):
    offset = 0.02 if row.mean_relative >= 0 else -0.02
    ha = "left" if row.mean_relative >= 0 else "right"
    ax.text(row.mean_relative + offset, i, f"{row.mean_relative:+.2f}",
            va="center", ha=ha, fontsize=9, color="#333")

ax.set_xlabel("\u2190 Below patient's average     Relative sentiment     Above patient's average \u2192", fontsize=10)
ax.set_title("Within-Patient Treatment Comparison (POTS patients)\nMean relative score vs each patient's own average", fontsize=12)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

**What this means:** This analysis controls for between-patient differences by comparing each patient's response to a treatment against that same patient's average across all treatments tried. Treatments with positive relative scores are rated better than the person's own baseline, suggesting genuine treatment-specific benefit rather than patient-level optimism or pessimism.

Magnesium, electrolytes, and probiotics consistently outperform each patient's own average, while famotidine and nattokinase fall below. This within-patient signal is arguably stronger evidence than group-level comparisons because it eliminates confounders like illness severity, reporting style, and demographic differences.

## Recommendations

Treatments are tiered by evidence strength based on sample size and statistical significance among POTS patients in this dataset.

In [ ]:
# --- Build tiered recommendations ---
rec_data = pots_drug_summary[
    (pots_drug_summary["n_users"] >= 3) &
    (pots_drug_summary["pct_pos"] >= 50)
].copy()

def assign_tier(row):
    if row["n_users"] >= 7 and row["pct_pos"] >= 70:
        return "Strong"
    elif row["n_users"] >= 5 or (row["n_users"] >= 3 and row["pct_pos"] >= 75):
        return "Moderate"
    else:
        return "Preliminary"

rec_data["tier"] = rec_data.apply(assign_tier, axis=1)
rec_data = rec_data.sort_values(["tier", "pct_pos"], ascending=[True, False])

# Annotations for each treatment
notes = {
    "magnesium": "Most consistent positive signal across all POTS subgroups. Supports muscle and nerve function.",
    "low dose naltrexone": "Strong positive rate in this sample. Broadly positive across POTS and non-POTS alike.",
    "electrolyte": "Core POTS management strategy. Consistently above patient's own average.",
    "probiotics": "Strong gut-health signal. Especially positive in MCAS and ME/CFS overlap.",
    "ketotifen": "Mast-cell stabilizer; useful for POTS+MCAS patients.",
    "antihistamines": "Positive overall; response varies by POTS subgroup.",
    "n-acetylcysteine": "Antioxidant with positive signal. Small sample.",
    "vitamin d": "Broadly positive. Common in post-viral recovery protocols.",
    "coq10": "Mixed results. Positive in some subgroups, negative in others.",
    "ssri": "Modest signal. Use context-dependent (some for anxiety/dysautonomia).",
    "red light therapy": "100% positive but very small sample (n=4).",
    "ivabradine": "Heart rate control for POTS. 100% positive (n=3).",
    "promethazine": "Antihistamine. 100% positive (n=3).",
    "potassium": "Electrolyte supplement. 100% positive (n=3).",
    "quercetin": "Natural mast-cell stabilizer. 75% positive (n=4).",
    "vitamin c": "Antioxidant. 100% positive (n=3).",
}

# Display as styled HTML table
html = "<table style='border-collapse:collapse; width:100%;'>"
html += "<tr style='background:#f0f0f0;'><th style='padding:8px; text-align:left;'>Tier</th><th style='padding:8px; text-align:left;'>Treatment</th><th style='padding:8px;'>Users</th><th style='padding:8px;'>% Positive</th><th style='padding:8px; text-align:left;'>Note</th></tr>"

tier_order = ["Strong", "Moderate", "Preliminary"]
tier_bg = {"Strong": "#d5f5e3", "Moderate": "#fef9e7", "Preliminary": "#f2f4f4"}

for tier in tier_order:
    tier_rows = rec_data[rec_data["tier"] == tier]
    for _, row in tier_rows.iterrows():
        bg = tier_bg[tier]
        note = notes.get(row["drug"], "")
        html += f"<tr style='background:{bg};'>"
        html += f"<td style='padding:6px 8px; font-weight:bold; color:{TIER_COLORS[tier]};'>{tier}</td>"
        html += f"<td style='padding:6px 8px;'>{row['drug'].title()}</td>"
        html += f"<td style='padding:6px 8px; text-align:center;'>{row['n_users']}</td>"
        html += f"<td style='padding:6px 8px; text-align:center;'>{row['pct_pos']:.0f}%</td>"
        html += f"<td style='padding:6px 8px; font-size:0.9em; color:#555;'>{note}</td>"
        html += "</tr>"

html += "</table>"
display(HTML(html))

In [ ]:
# --- Visual summary: forest plot of recommended treatments, color-coded by tier ---
rec_plot = rec_data.sort_values("pct_pos").copy()

fig, ax = plt.subplots(figsize=(11, max(5, len(rec_plot) * 0.4)))
y = np.arange(len(rec_plot))

# Calculate CI for each
rec_means, rec_cis = [], []
for _, row in rec_plot.iterrows():
    scores = pots_user_drug[pots_user_drug["drug"] == row["drug"]]["avg_score"].values
    m = scores.mean()
    se = sp_stats.sem(scores) if len(scores) > 1 else 0
    ci = se * sp_stats.t.ppf(0.975, max(len(scores) - 1, 1)) if len(scores) > 1 else 0
    rec_means.append(m)
    rec_cis.append(ci)

dot_colors = [TIER_COLORS[t] for t in rec_plot["tier"]]

ax.errorbar(rec_means, y, xerr=rec_cis, fmt="none", ecolor="#888",
            elinewidth=1.5, capsize=5, zorder=1)
ax.scatter(rec_means, y, c=dot_colors, s=120, zorder=2,
           edgecolors="white", linewidths=1)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.4)

ax.set_yticks(y)
ax.set_yticklabels(
    [f"{row.drug.title()} (n={row.n_users})" for row in rec_plot.itertuples()],
    fontsize=10
)

for i, (m, ci) in enumerate(zip(rec_means, rec_cis)):
    ax.text(m + ci + 0.05, i, f"{m:.2f}", va="center", fontsize=9, color="#333")

ax.set_xlabel("Mean sentiment score (95% CI)", fontsize=11)
ax.set_title("Recommended Treatments for POTS Patients\n(color-coded by evidence tier)", fontsize=13)
ax.set_xlim(-0.6, 1.6)

# Legend for tiers
handles = [mpatches.Patch(color=TIER_COLORS[t], label=f"{t} evidence") for t in tier_order]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.08),
          ncol=3, frameon=False, fontsize=10)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.subplots_adjust(bottom=0.15)
plt.tight_layout()
plt.show()

## Summary

### Key Findings

**Do POTS patients respond differently to treatments?**

Yes, with caveats. POTS patients show a distinct treatment preference profile compared to the broader Long COVID population:

- **Magnesium** is the top-performing treatment among POTS patients (100% positive, n=7), consistent across all comorbidity subgroups. This aligns with its role in nerve and muscle function, relevant to autonomic dysfunction.
- **Electrolytes** (83% positive, n=6) and **probiotics** (83%, n=6) are also strong performers, consistent with standard POTS management (volume expansion) and emerging gut-brain axis research.
- **LDN** shows 86% positive outcomes among POTS users (n=7), similar to its overall 62% positive rate across all users. The difference is not statistically significant (Mann-Whitney p=0.84), and the sample is too small for definitive conclusions.
- **Beta blockers** — a standard POTS medication — paradoxically show poor outcomes (25% positive), likely reflecting that patients discuss frustrations and side effects rather than routine maintenance.

**How does LDN compare across subgroups?**

LDN has limited data within POTS subgroups (only 5–7 users per subgroup who also tried LDN), making formal subgroup comparisons unreliable. At the broader treatment landscape level:

- **POTS+MCAS** patients (n=60): Best responses to magnesium, electrolytes, probiotics, and ketotifen (mast-cell stabilizer)
- **POTS+ME/CFS** patients (n=48): Similar pattern, with antihistamines and vitamin D also performing well
- **POTS+EDS** patients (n=27): More variable responses; antihistamines underperform compared to other subgroups

### Caveats

- **Small samples**: Most treatments have 3–13 POTS users. Results should be treated as hypothesis-generating, not confirmatory.
- **Selection bias**: POTS patients who post about treatments are not representative of all POTS patients.
- **Reporting bias**: Positive outcomes may be over-reported if patients are more motivated to share what helped.
- **Comorbidity overlap**: 96% of POTS users also have Long COVID; subgroups overlap heavily. Isolated POTS effects cannot be cleanly separated.

### Suggested Next Steps

1. Expand data collection to 3–6 months for more robust per-treatment sample sizes within POTS
2. Investigate the beta-blocker paradox with text-level analysis (are negative reports about side effects or inefficacy?)
3. Run multi-treatment pathway analysis for POTS patients who tried sequential treatments

---

*Based on self-selected Reddit posts. Users who never posted about a treatment are not represented. Results reflect reporting patterns, not population-level treatment effects.*

In [ ]:
conn.close()